In [5]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import plotly.express as px
from seaborn import heatmap
from sklearn.cluster import HDBSCAN

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder
)

In [62]:
from sklearn.utils.validation import check_random_state
check_random_state(42)

RandomState(MT19937) at 0x73ABCBC1E540

In [4]:
DATA_DIR = '../../data/raw'
DATASET = f'{DATA_DIR}/dataset_practica_final.csv'

In [34]:
dset = pd.read_csv(DATASET)
dset_cp = dset.copy()

In [35]:
dset_cp['country'] = dset_cp['country'].fillna('UNKNOWN')

In [48]:
country_frequency = pd.DataFrame(dset['country'].value_counts(normalize=True)).reset_index()

In [52]:
dset_cp = dset_cp.merge(country_frequency, on='country')

/tmp/ipykernel_5487/451231498.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dset_cp[dset_cp.proportion < 0.05]['country'] = 'OTROS'


In [67]:
def loader(dataset_csv_filename:str) -> (pd.core.frame.DataFrame, pd.core.frame.DataFrame, pd.core.series.Series, pd.core.series.Series):
    
    # se setea random_state para que los resultados sean predecibles (que el resultado sea el mismo siempre)
    check_random_state(42)

    # carga del csv en un dataframe
    dset = pd.read_csv(DATASET)
    
    # eliminación de las columnas conflictivas
    columns_to_drop =[
                        'reservation_status',
                        'reservation_status_date',
                        'arrival_date_year',
                        'arrival_date_month',
                        'arrival_date_day_of_month',
                        'previous_bookings_not_canceled']

    # separación en target y features
    y = dset['is_canceled']
    X = dset.drop(['is_canceled'], axis=1)
    dset.drop(columns_to_drop, axis=1, inplace=True)

    # OneHot encoder de las variables categóricas (no ordinales) y standa
    list_one_hot_cols = [
                            'hotel',
                            'meal',
                            #'country',
                            'market_segment',
                            'distribution_channel',
                            'is_repeated_guest',
                            'reserved_room_type',
                            'assigned_room_type',
                            'deposit_type',
                            'agent',
                            'company',
                            'customer_type']

    list_numeric_cols = [
                            'lead_time',
                            'stays_in_weekend_nights',
                            'stays_in_week_nights',
                            'adults',
                            'children',
                            'babies',
                            'previous_cancellations',
                            'booking_changes',
                            'days_in_waiting_list',
                            'adr',
                            'required_car_parking_spaces',
                            'total_of_special_requests']

    columns_preprocessor = ColumnTransformer(transformers=[
        ('one_hot', OneHotEncoder(drop='first', sparse_output=False), list_one_hot_cols),
        #('ordinal', OrdinalEncoder(), list_ordinal_cols),
        ('numeric', MinMaxScaler(), list_numeric_cols)
    ])
    columns_preprocessor.fit_transform(X)

    # relleno de los valores faltantes
    X['children'] = X['children'].fillna(X['children'].median())
    X['agent'] = X['agent'].fillna(0)
    X['company'] = X['company'].fillna(0)

    # países con poca frecuencia se clasifican como 'otros'
    # PENDIENTE

    # test - train split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)  

    # eliminación de outlayers del conjunto de train
    # hdbscan = HDBSCAN(min_cluster_size=5,
    #                   cluster_selection_epsilon=0.6,
    #                   metric='euclidean',algorithm='auto',
    #                   n_jobs=-1,
    #                   cluster_selection_method='eom')
    # clusters = hdbscan.fit(X_train)
    # X_train['cluster'] = clusters.labels_
    # X_train = X_train[X_train.cluster!=-1].drop(['cluster'], axis=1)   

    return (X_train, X_test, y_train, y_test)

In [65]:
X_train, X_test, y_train, y_test = loader(DATASET)

ValueError: A given column is not a column of the dataframe